# Task 1 — Exploratory Data Analysis (EDA)

**Dataset:** Sample Superstore  
**Tools:** Python, Pandas, Matplotlib, Seaborn, SciPy

Explore dataset structure and quality, identify patterns and anomalies, test statistical relationships, and highlight issues relevant to further analysis.

## 1. Analysis Questions

- What is the structure and quality of the dataset?
- Which variables are numerical and categorical?
- How are sales and profit distributed?
- Which categories and regions contribute most to sales and profit?
- Is discount associated with profit?
- Does profit distribution differ across categories?
- Are there potential outliers or data-quality issues?
- Does the dataset contain a time variable for trend analysis?

## 2. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
PRIMARY = "#5B5BD6"
ACCENT = "#F4A261"

## 3. Load and Inspect Data

In [ ]:
df = pd.read_csv("../data/SampleSuperstore_cleaned.csv")

print(f"Dataset shape: {df.shape}")
display(df.head())

## 4. Structure and Data Types

In [ ]:
structure = pd.DataFrame({
    "Data Type": df.dtypes.astype(str),
    "Unique Values": df.nunique(),
    "Missing Values": df.isna().sum()
})

display(structure)

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(exclude=np.number).columns.tolist()

print("Numerical variables:", numeric_cols)
print("Categorical variables:", categorical_cols)

## 5. Data Quality

In [ ]:
quality = pd.DataFrame({
    "Missing Values": df.isna().sum(),
    "Missing %": (df.isna().mean() * 100).round(2),
    "Unique Values": df.nunique(),
    "Data Type": df.dtypes.astype(str)
})

print(f"Missing values: {df.isna().sum().sum()}")
print(f"Exact duplicate rows: {df.duplicated().sum()}")
display(quality)

## 6. Descriptive Statistics

In [ ]:
display(df.describe().T)

summary = pd.DataFrame({
    "Metric": ["Total Sales", "Total Profit", "Total Quantity", "Average Discount", "Profit Margin"],
    "Value": [
        df["Sales"].sum(),
        df["Profit"].sum(),
        df["Quantity"].sum(),
        df["Discount"].mean(),
        df["Profit"].sum() / df["Sales"].sum()
    ]
})

display(summary)

## 7. Sales and Profit Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.histplot(df["Sales"], bins=40, kde=True, color=PRIMARY, ax=axes[0])
axes[0].set_title("Sales Distribution", fontweight="bold")
axes[0].set_xlabel("Sales")
axes[0].set_ylabel("Order Count")

sns.histplot(df["Profit"], bins=40, kde=True, color=ACCENT, ax=axes[1])
axes[1].axvline(0, linestyle="--", linewidth=1.5)
axes[1].set_title("Profit Distribution", fontweight="bold")
axes[1].set_xlabel("Profit")
axes[1].set_ylabel("Order Count")

plt.tight_layout()
plt.show()

## 8. Category and Regional Patterns

In [ ]:
category_summary = df.groupby("Category")[["Sales", "Profit"]].sum().sort_values("Profit", ascending=False)
region_summary = df.groupby("Region")[["Sales", "Profit"]].sum().sort_values("Profit", ascending=False)

display(category_summary)
display(region_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

category_summary[["Sales", "Profit"]].plot(kind="bar", ax=axes[0])
axes[0].set_title("Sales and Profit by Category", fontweight="bold")
axes[0].set_xlabel("Category")
axes[0].set_ylabel("Amount")
axes[0].tick_params(axis="x", rotation=0)
axes[0].legend(title="")

region_summary["Profit"].sort_values().plot(kind="barh", ax=axes[1])
axes[1].set_title("Profit by Region", fontweight="bold")
axes[1].set_xlabel("Profit")
axes[1].set_ylabel("Region")

plt.tight_layout()
plt.show()

## 9. Hypothesis Test — Discount vs Profit

**H₀:** Discount and profit have no monotonic relationship.  
**H₁:** Discount and profit have a monotonic relationship.

Spearman correlation is used because it does not require normally distributed variables.

In [ ]:
rho, p_value = stats.spearmanr(df["Discount"], df["Profit"])

print(f"Spearman correlation: {rho:.4f}")
print(f"P-value: {p_value:.3e}")
print("Decision:", "Reject H₀" if p_value < 0.05 else "Fail to reject H₀")

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="Discount", y="Profit", alpha=0.35, s=25, color=PRIMARY)
plt.axhline(0, linestyle="--", linewidth=1.2)
plt.title("Discount vs Profit", fontweight="bold")
plt.xlabel("Discount")
plt.ylabel("Profit")
plt.tight_layout()
plt.show()

## 10. Hypothesis Test — Profit Across Categories

**H₀:** Profit distributions are the same across categories.  
**H₁:** At least one category has a different profit distribution.

The Kruskal–Wallis test is used as a non-parametric alternative to one-way ANOVA.

In [ ]:
category_groups = [group["Profit"].values for _, group in df.groupby("Category")]
statistic, p_value = stats.kruskal(*category_groups)

print(f"Kruskal-Wallis statistic: {statistic:.4f}")
print(f"P-value: {p_value:.3e}")
print("Decision:", "Reject H₀" if p_value < 0.05 else "Fail to reject H₀")

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="Category", y="Profit")
plt.axhline(0, linestyle="--", linewidth=1.2)
plt.title("Profit Distribution Across Categories", fontweight="bold")
plt.xlabel("Category")
plt.ylabel("Profit")
plt.tight_layout()
plt.show()

## 11. Outlier Detection

In [ ]:
outlier_rows = []

for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_rows.append([col, q1, q3, lower, upper, int(count)])

outlier_report = pd.DataFrame(
    outlier_rows,
    columns=["Variable", "Q1", "Q3", "Lower Bound", "Upper Bound", "Outlier Count"]
)

display(outlier_report)

## 12. Key Findings and Data Issues

- The supplied dataset contains **9,977 rows and 13 variables**.
- No missing values or exact duplicate rows are present in the supplied cleaned dataset.
- Sales are right-skewed, while profit contains both positive and negative observations.
- Category and regional summaries reveal differences in sales and profitability.
- Discount shows a statistically significant negative monotonic relationship with profit in this dataset.
- Profit distributions differ significantly across categories according to the Kruskal–Wallis test.
- IQR analysis flags potential extreme observations that should be investigated rather than automatically removed.
- The dataset has **no date/time variable**, so temporal trend analysis cannot be performed from this file alone.
- Negative profit values should be retained because they represent meaningful business outcomes.

## 13. Conclusion

This EDA documents the dataset structure and quality, identifies important business patterns and potential anomalies, validates key relationships statistically, and highlights considerations for further analysis or modeling.